In [13]:

import joblib

joblib.dump(best_model, "best_decision_tree_churn.pkl")

print("Best Decision Tree model saved successfully!")


Best Decision Tree model saved successfully!


In [12]:

tuned_train_pred = best_model.predict(X_train)
tuned_test_pred = best_model.predict(X_test)

comparison = pd.DataFrame([
    {
        "Model": "Original Decision Tree",
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, zero_division=0),
        "Recall": recall_score(y_test, test_pred, zero_division=0),
        "F1 Score": f1_score(y_test, test_pred, zero_division=0)
    },
    {
        "Model": "Tuned Decision Tree",
        "Train Accuracy": accuracy_score(y_train, tuned_train_pred),
        "Test Accuracy": accuracy_score(y_test, tuned_test_pred),
        "Precision": precision_score(y_test, tuned_test_pred, zero_division=0),
        "Recall": recall_score(y_test, tuned_test_pred, zero_division=0),
        "F1 Score": f1_score(y_test, tuned_test_pred, zero_division=0)
    }
])

print(comparison.round(4))


                    Model  Train Accuracy  Test Accuracy  Precision  Recall  \
0  Original Decision Tree          0.9980         0.7296     0.4908  0.4973   
1     Tuned Decision Tree          0.8301         0.7935     0.6334  0.5267   

   F1 Score  
0    0.4940  
1    0.5752  


In [11]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1 Score:", grid_search.best_score_)


Fitting 5 folds for each of 96 candidates, totalling 480 fits
Best Parameters: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 10}
Best Cross-Validation F1 Score: 0.5641008812483899


In [10]:

original_model = DecisionTreeClassifier(random_state=42)

original_model.fit(X_train, y_train)

train_pred = original_model.predict(X_train)
test_pred = original_model.predict(X_test)

print("ORIGINAL DECISION TREE RESULTS")
print("--------------------------------")
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("Testing Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred, zero_division=0))
print("Recall:", recall_score(y_test, test_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, test_pred, zero_division=0))


ORIGINAL DECISION TREE RESULTS
--------------------------------
Training Accuracy: 0.9980475683351083
Testing Accuracy: 0.7295954577714692
Precision: 0.49076517150395776
Recall: 0.49732620320855614
F1 Score: 0.4940239043824701


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


Training features: (5634, 30)
Testing features: (1409, 30)


In [14]:
# Train the original Decision Tree
original_model = DecisionTreeClassifier(random_state=42)

original_model.fit(X_train, y_train)

# Predictions
train_pred = original_model.predict(X_train)
test_pred = original_model.predict(X_test)

# Evaluate original model
print("ORIGINAL DECISION TREE RESULTS")
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)
print("--------------------------------")
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("Testing Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1 Score:", f1_score(y_test, test_pred))

ORIGINAL DECISION TREE RESULTS
Training features: (5634, 30)
Testing features: (1409, 30)
Training targets: (5634,)
Testing targets: (1409,)
--------------------------------
Training Accuracy: 0.9980475683351083
Testing Accuracy: 0.7295954577714692
Precision: 0.49076517150395776
Recall: 0.49732620320855614
F1 Score: 0.4940239043824701


In [6]:
# Separate target
df = df.dropna(subset=["Churn Label"]).copy()

y = df["Churn Label"].map({"Yes": 1, "No": 0})

# Remove unnecessary geographic columns and target
drop_cols = [
    "Churn Label",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "CLTV"
]

X = df.drop(columns=drop_cols, errors="ignore").copy()

# Convert numeric columns to numeric types
numeric_cols = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges"
]

for col in numeric_cols:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# Encode categorical columns only
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

# Handle missing and infinite values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Data types:")
print(X.dtypes.value_counts())

Features shape: (7043, 30)
Target shape: (7043,)
Data types:
int64      28
float64     2
Name: count, dtype: int64


In [5]:
for col in df.columns:
    print(col, ":", df[col].nunique(), "unique values")

City : 1129 unique values
Zip Code : 1652 unique values
Lat Long : 1652 unique values
Latitude : 1652 unique values
Longitude : 1651 unique values
Gender : 2 unique values
Senior Citizen : 2 unique values
Partner : 2 unique values
Dependents : 2 unique values
Tenure Months : 73 unique values
Phone Service : 2 unique values
Multiple Lines : 3 unique values
Internet Service : 3 unique values
Online Security : 3 unique values
Online Backup : 3 unique values
Device Protection : 3 unique values
Tech Support : 3 unique values
Streaming TV : 3 unique values
Streaming Movies : 3 unique values
Contract : 3 unique values
Paperless Billing : 2 unique values
Payment Method : 4 unique values
Monthly Charges : 1585 unique values
Total Charges : 6531 unique values
Churn Label : 2 unique values
CLTV : 3438 unique values


In [4]:
# Separate target
df = df.dropna(subset=["Churn Label"]).copy()
y = df["Churn Label"].map({"Yes": 1, "No": 0})

# Remove target and identifier / leakage columns
columns_to_drop = [
    "Churn Label",
    "CLTV",
    "Lat Long",
    "City",
    "Zip Code",
    "Latitude",
    "Longitude",
    "CustomerID",
    "Customer ID"
]

X = df.drop(
    columns=[c for c in columns_to_drop if c in df.columns]
)

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Handle missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts())

Features shape: (7043, 6559)
Target shape: (7043,)
Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [3]:
# Separate the target column
df = df.dropna(subset=["Churn Label"]).copy()

y = df["Churn Label"].map({"Yes": 1, "No": 0})

# Remove target and columns that could cause data leakage
X = df.drop(columns=[
    "Churn Label",
    "CLTV",
    "Lat Long",
    "City",
    "Zip Code"
], errors="ignore")

# Convert categorical columns into numbers
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Handle missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median()).fillna(0)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nChurn distribution:")
print(y.value_counts())

Features shape: (7043, 6561)
Target shape: (7043,)

Churn distribution:
Churn Label
0    5174
1    1869
Name: count, dtype: int64


In [2]:
df = pd.read_csv("../data/cleaned_churn_data.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (7043, 26)
          City  Zip Code                Lat Long   Latitude   Longitude  \
0  Los Angeles     90003  33.964131, -118.272783  33.964131 -118.272783   
1  Los Angeles     90005   34.059281, -118.30742  34.059281 -118.307420   
2  Los Angeles     90006  34.048013, -118.293953  34.048013 -118.293953   
3  Los Angeles     90010  34.062125, -118.315709  34.062125 -118.315709   
4  Los Angeles     90015  34.039224, -118.266293  34.039224 -118.266293   

   Gender Senior Citizen Partner Dependents  Tenure Months  ... Tech Support  \
0    Male             No      No         No              2  ...           No   
1  Female             No      No        Yes              2  ...           No   
2  Female             No      No        Yes              8  ...           No   
3  Female             No     Yes        Yes             28  ...          Yes   
4    Male             No      No        Yes             49  ...           No   

  Streaming TV Streaming Movies        Con

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries imported successfully!")

Libraries imported successfully!
